<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/3_Analisis_y_visualizacion_datos/5_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# 🔎📊**Análisis Exploratorio de Datos (Exploratory Data Analysis - EDA)**

👉 Con datos listos, podemos descubrir patrones.

✅ Resultado: entendemos la historia detrás de los datos y generamos hipótesis.

**Objetivos de Aprendizaje:**

Al finalizar este notebook, serás capaz de:

1. **Diagnosticar un conjunto de datos:** Utilizar funciones de Pandas para obtener un resumen estadístico y estructural de los datos, identificando tipos de variables, valores nulos y dimensiones.

2. **Analizar y visualizar distribuciones:** Crear e interpretar gráficos univariados (histogramas, boxplots) para entender la distribución, tendencia central y dispersión de las variables económicas.

3. **Descubrir relaciones entre variables:** Implementar y analizar gráficos multivariados (scatter plots, heatmaps) para identificar correlaciones, covarianzas y patrones entre múltiples indicadores económicos.

**Introducción**

Imaginemos que la OCDE nos contrata para entender los factores que determinan el desarrollo económico de los países. Antes de lanzarnos a construir un complejo modelo econométrico con regresiones y series de tiempo, un buen economista haría un "trabajo de campo": viajaría (conceptualmente) a varios países, observaría sus economías, hablaría con la gente y recolectaría datos preliminares.

El Análisis Exploratorio de Datos (EDA) es precisamente eso, es el proceso de usar estadísticas descriptivas y visualizaciones para "dialogar" con nuestros datos.

Queremos que los datos nos cuenten su historia:

- ¿Cómo es un país "promedio" en nuestra muestra? (Media, mediana)

- ¿Qué tan desigual es el mundo en términos de ingreso o salud? (Desviación estándar, percentiles, boxplots)

- ¿Existen "clubes de convergencia" o grupos de países con características similares? (Análisis de categorías, gráficos de densidad)

- ¿Parece haber una relación entre la inversión en educación y el crecimiento del PIB? (Gráficos de dispersión, correlación)

Hacer un buen EDA nos permite formular mejores hipótesis, detectar problemas de calidad en los datos (como outliers o valores faltantes) y construir modelos más robustos y significativos. Es el arte de hacer las preguntas correctas a nuestros datos antes de pedirles que nos den respuestas definitivas.

## Importar librerías

In [ ]:
# En VS Code, si no has instalado las librerías, ejecutar:
# %pip install matplotlib
# %pip install seaborn
# %pip install plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

## Mejorar visualización de los dataframes y los gráficos

In [ ]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
# Configuraciones para una mejor visualización de los gráficos
sns.set_style("whitegrid")

## Cargar el dataset

Usaremos un dataset de **Gapminder** (https://www.gapminder.org/) que viene precargado en Plotly Express para facilitar el acceso. Contiene datos de PIB per cápita PPA a precios constantes, esperanza de vida y población para varios países a lo largo del tiempo.

**Nota**

Gapminder es una fundación educativa independiente y sin fines de lucro. Fue cofundada en Suecia por el legendario médico y estadístico Hans Rosling, junto a su hijo y su nuera.

La misión de la Fundación Gapminder es luchar contra las ideas erróneas y preconcebidas sobre el desarrollo global. Rosling sostenía que la mayoría de la gente, incluso personas muy educadas y en posiciones de poder, tienen una visión del mundo anticuada, dividiendo a los países en "desarrollados" y "en vías de desarrollo", una dicotomía que los datos ya no respaldan.

El método de Gapminder para combatir esta desinformación es el uso magistral de la visualización de datos para contar historias (data storytelling).

In [ ]:
df = px.data.gapminder()

## Exploración inicial del dataset

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.info()

**Interpretación de los resultados:**

Vemos que tenemos 1704 observaciones (filas) y 8 variables (columnas). Afortunadamente, no hay valores nulos, lo cual es ideal pero poco común en el mundo real. Las variables como esperanza_vida, poblacion y pib_per_capita son numéricas, mientras que país y continente son tipo object.

In [ ]:
# vamos a renombrar las columnas para un mejor entendimiento
df.rename(columns={'lifeExp':'esperanza_vida',
                   'pop':'poblacion',
                   'gdpPercap':'pib_per_capita',
                   'country':'pais',
                   'continent':'continente',
                   'year':'año'},
          inplace=True)
df

In [ ]:
df['año'] = df['año'].astype('str')

In [ ]:
df.drop(columns=['iso_alpha', 'iso_num'], inplace=True)

In [ ]:
df.head()

## Análisis de variables no numéricas

### Variables categóricas

Para las variables categóricas, nos interesa saber cuántas categorías únicas existen, cuáles son y cómo se distribuyen las observaciones entre ellas.

In [ ]:
df['continente']

In [ ]:
df['continente']

In [ ]:
print("Conteo de observaciones por continente:")
print(df['continente'].value_counts())

In [ ]:
print("\nPorcentaje de observaciones por continente:")
print(df['continente'].value_counts() * 100)

Asia y África son los continentes con más observaciones en nuestra muestra, lo que tiene sentido ya que tienen un gran número de países. Oceanía tiene la menor cantidad.

### Análisis de los individuos objeto de estudio

¿Qué países se encuentran en la base de datos?

In [ ]:
df['pais'].unique()

¿Cuántos países son?

In [ ]:
df['pais'].nunique()

In [ ]:
df['pais'].value_counts()

### Análisis de la dimensión temporal

¿Qué años se encuentran en la base de datos?

In [ ]:
df['año'].unique()

¿Cuántos años son?

In [ ]:
df['año'].nunique()

In [ ]:
df['año'].value_counts()

## Estadísticas Descriptivas (Análisis Univariado)

Ahora, vamos a calcular las estadísticas fundamentales para nuestras variables numéricas.

In [ ]:
print("Estadísticas Descriptivas de las Variables Numéricas:")


**Trabajo en parejas**

Realicen y compartan con la clase un análisis de estas estadísticas para esperanza de vida, población y PIB per cápita. Tiempo: 2 minutos.

![asimetria](https://drive.google.com/uc?id=1M4qWRXBzUIVTNFSl5Yw2vfMppr1DQYXe)

También podemos calcular estas y otras medidas de forma individual.

In [ ]:
# Conteo de los valores no nulos de la columna
df['pib_per_capita']

In [ ]:
# Mínimo
df['pib_per_capita']

In [ ]:
# Máximo
df['pib_per_capita']

In [ ]:
# Media
df['pib_per_capita']

In [ ]:
# Desviación estándar
df['pib_per_capita']

In [ ]:
# Mediana
df['pib_per_capita']

In [ ]:
# Percentil 50 (mediana)
df['pib_per_capita']

In [ ]:
# Percentil 25 y 75
df['pib_per_capita']

In [ ]:
# Varios percentiles a la vez
df['pib_per_capita']

In [ ]:
# Si quieres todos los deciles (10%, 20%, …, 90%)
df['pib_per_capita'].quantile([i/10 for i in range(1, 10)])

In [ ]:
# Mostrar los 20 valores más altos
df['pib_per_capita']

In [ ]:
# Mostrar los 5 valores más bajos
df['pib_per_capita']

In [ ]:
# Si quieres las filas completas del DataFrame que corresponden a esos 5 valores
df.nsmallest(5, 'pib_per_capita')

In [ ]:
# Moda
df['esperanza_vida']

In [ ]:
# Verificación
df['esperanza_vida'].value_counts()

## Ejercicio

Calcula las estadísticas descriptivas solo para los países del continente asiático (Asia) en el año 2007. ¿Cómo se compara el PIB per cápita medio de Asia en 2007 con la media global de toda la muestra que calculamos antes?

La media del PIB per cápita en Asia en 2007 (USD 12.473) es considerablemente más alta que la media global de toda la muestra histórica (USD 7.215).

## Visualización de Datos (Análisis Gráfico)

Ahora vamos a graficar nuestros datos para entender las historias que nos contaron las estadísticas descriptivas.

**Nota**

Para mayor información, consulte la documentación de las librerías:

- Matplotlib: https://matplotlib.org/

- Seaborn (enfocado en gráficos estadísticos): https://seaborn.pydata.org/index.html

- Plotly (para gráficos interactivos): https://plotly.com/python/

### Visualizaciones Univariadas (Una variable)

Estos gráficos nos ayudan a entender la distribución de una sola variable.

#### Histograma y Gráfico de Densidad

¿Cómo se distribuye el PIB?

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df['pib_per_capita'], kde=True, bins=50)
plt.title('Distribución del PIB per cápita global')
plt.xlabel('PIB per Cápita (USD PPA a precios constantes)')
plt.ylabel('Frecuencia')
plt.show()

**Ejercicio**: Interprete el gráfico.

#### Boxplot: Identificando la Dispersión y los Outliers

El boxplot es una de las herramientas más poderosas para un analista. Resume la distribución en 5 números (mínimo, Q1, mediana, Q3, máximo) y visualiza los outliers.

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(y='continente', x='esperanza_vida', data=df)
plt.title('Boxplot de Esperanza de Vida por Continente')
plt.xlabel('Esperanza de Vida (Años)')
plt.ylabel('Continente')
plt.show()

**Ejercicio**: Interprete el gráfico.

#### Gráfico de Torta

Un gráfico de torta (o pie chart) se usa para mostrar la proporción de cada categoría sobre un total. Es visualmente atractivo, pero como veremos, puede ser más difícil de interpretar con precisión que un gráfico de barras.

In [ ]:
# 1. Preparar los datos: Contar las observaciones por continente
conteo_continentes = df['continente'].value_counts()
conteo_continentes

In [ ]:
# 2. Crear el gráfico de torta
plt.figure(figsize=(10, 10))
plt.pie(conteo_continentes,
        labels=conteo_continentes.index,
        autopct='%1.1f%%', # Formato para mostrar porcentajes con un decimal
        startangle=90      # Inicia el primer 'trozo' en la parte superior
        )

plt.title('Proporción de Observaciones por Continente', fontsize=16)
plt.ylabel('') # Quitamos el label del eje y que matplotlib añade por defecto
plt.show()

**Ejercicio**: Interprete el gráfico.

**Nota**

Usa un gráfico de torta solo cuando tengas pocas categorías (idealmente, menos de 5) y tu objetivo principal sea destacar la proporción de una de esas categorías con respecto al total, más que comparar las categorías entre sí.

#### Gráfico de Barras

In [ ]:
# Gráfico de Barras
plt.figure(figsize=(10, 6))
sns.countplot(y='continente', data=df, order = df['continente'].value_counts().index)
plt.title('Número de Observaciones por Continente')
plt.xlabel('Conteo')
plt.ylabel('Continente')
plt.show()

**Ejercicio**: Interprete el gráfico.

El gráfico de barras es generalmente preferible al de torta, ya que es más fácil comparar longitudes que áreas. Confirma visualmente que África y Asia dominan nuestra muestra en términos de número de observaciones (país-año).

### Visualizaciones Multivariadas (Dos o más variables)

Aquí es donde empezamos a buscar relaciones.

#### Gráfico de Dispersión (Scatter Plot)

El scatter plot es la herramienta fundamental para visualizar la relación entre dos variables numéricas.

In [ ]:
# Tomemos solo los datos de 2007 para una foto más clara
df_2007 = df[df['año']=='2007']

plt.figure(figsize=(12, 7))
sns.scatterplot(x='pib_per_capita', y='esperanza_vida', data=df_2007)
plt.title('Esperanza de Vida vs. PIB per Cápita (2007)')
plt.xlabel('PIB per Cápita (USD PPA a precios constantes)')
plt.ylabel('Esperanza de Vida (Años)')
#plt.xscale('log') # Usamos escala logarítmica para el PIB por su gran sesgo
plt.show()

**Ejercicio**: Interprete el gráfico.

#### Gráfico de Burbujas: Añadiendo una Tercera Dimensión

Podemos añadir la variable poblacion al tamaño de los puntos para crear un gráfico de burbujas.

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='pib_per_capita', y='esperanza_vida', data=df_2007,
                size='poblacion', hue='continente', sizes=(20, 2000), alpha=0.7)
plt.title('Esperanza de Vida, PIB per Cápita y Población por Continente (2007)')
plt.xlabel('PIB per Cápita (USD PPA a precios constantes)')
plt.ylabel('Esperanza de Vida (Años)')
plt.xscale('log')
plt.legend(title='Continente y Población', bbox_to_anchor=(1.05, 1), loc=2)
plt.show()

**Ejercicio**: Interprete el gráfico.

#### Gráfico de Línea

In [ ]:
# Calculemos el PIB per cápita promedio por año y continente
pib_por_año_continente = df.groupby(['año', 'continente'])['pib_per_capita'].mean().reset_index()

plt.figure(figsize=(14, 8))
sns.lineplot(x='año', y='pib_per_capita', hue='continente', data=pib_por_año_continente, lw=3)
plt.title('Evolución del PIB per Cápita Promedio por Continente (1952-2007)')
plt.ylabel('PIB per Cápita Promedio (USD PPA a precios constantes)')
plt.xlabel('Año')
plt.legend(title='Continente')
plt.show()

**Ejercicio**: Interprete el gráfico.

#### Gráfico de Barras Apiladas

Mientras que un gráfico de líneas es bueno para comparar las tendencias de cada continente por separado, un gráfico de barras apiladas nos permite responder dos preguntas a la vez:

- ¿Cómo ha crecido la población total de nuestra muestra a lo largo de los años? (Altura total de la barra)

- ¿Cómo ha cambiado la participación porcentual de cada continente en esa población total? (Tamaño relativo de cada color dentro de la barra)

Para crearlo, primero necesitamos reorganizar (pivotar) nuestros datos para que los años sean el índice, los continentes sean las columnas y los valores sean la suma de la población.

In [ ]:
# 1. Preparar los datos: Crear una tabla pivote
# Agrupamos por año y continente, sumando la población.
# Luego, usamos .unstack() para convertir los continentes en columnas.
poblacion_continentes = df.groupby(['año', 'continente'])['poblacion'].sum().unstack()

# 2. Crear el gráfico de barras apiladas
ax = poblacion_continentes.plot(kind='bar', stacked=True, figsize=(16, 9),
                                 colormap='viridis')

# 3. Mejorar la legibilidad del gráfico
plt.title('Población por Continente a lo Largo del Tiempo (Gráfico Apilado)', fontsize=20)
plt.xlabel('Año', fontsize=14)
plt.ylabel('Población (en miles de millones)', fontsize=14)
plt.xticks(rotation=45)
plt.legend(title='Continente', bbox_to_anchor=(1.02, 1), loc='upper left')

# Formatear el eje Y para que muestre los números en miles de millones
# Esto hace que el gráfico sea mucho más fácil de leer.
formatter = lambda x, pos: f'{x/1e9:.1f}B'
ax.yaxis.set_major_formatter(formatter)

plt.tight_layout()
plt.show()

**Ejercicio**: Interprete el gráfico.

## Correlación

Los gráficos de dispersión sugieren relaciones. La correlación las cuantifica numéricamente.

- Correlación > 0: Relación positiva (cuando X sube, Y tiende a subir).

- Correlación < 0: Relación negativa (cuando X sube, Y tiende a bajar).

- Correlación ≈ 0: No hay relación lineal.

Un **heatmap** es una forma excelente de visualizar la matriz de correlación, usando colores para representar los valores.

In [ ]:
# Seleccionamos solo las columnas numéricas para la correlación
df_numeric = df[['esperanza_vida', 'poblacion', 'pib_per_capita']]
correlation_matrix = df_numeric.corr()
correlation_matrix

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlación de Variables Numéricas')
plt.show()

**Ejercicio**: Interprete el gráfico.

⚠️ Advertencia clave: **¡Correlación no implica causalidad!** El hecho de que dos variables se muevan juntas no significa que una cause la otra. Podría haber una tercera variable oculta (variable omitida) o la causalidad podría ir en la dirección opuesta.

**Ejercicio de discusión**

Hemos visto una correlación de 0.58 entre PIB per cápita y esperanza de vida. Pero, ¿es una prueba de que 'el dinero compra la salud'? Piensen por un minuto en al menos una tercera variable (una variable omitida) que podría estar causando que tanto el PIB como la esperanza de vida se muevan juntas.

## Gráficos Avanzados con Seaborn

Seaborn ofrece herramientas de alto nivel para explorar relaciones complejas con poco código.

### Pairplot: Todas las Relaciones de un Vistazo

El pairplot crea una matriz de gráficos: histogramas en la diagonal (para ver la distribución de cada variable) y scatter plots en el resto (para ver la relación entre cada par de variables). Es una forma increíblemente eficiente de obtener una visión general.

In [ ]:
# Usamos una muestra de los datos para que el gráfico se genere más rápido
df_sample = df.sample(500)

sns.pairplot(df_sample, diag_kind='kde')
plt.suptitle('Pairplot de Indicadores (Muestra)', y=1.02)
plt.show()

In [ ]:
# Usamos una muestra de los datos para que el gráfico se genere más rápido
df_sample = df.sample(500)

sns.pairplot(df_sample, hue='continente', diag_kind='kde')
plt.suptitle('Pairplot de Indicadores por Continente (Muestra)', y=1.02)
plt.show()

Este único comando genera una gran cantidad of información. Podemos ver todas las relaciones bivariadas simultáneamente, segmentadas por color según el continente. Rápidamente podemos confirmar visualmente las correlaciones que vimos en el heatmap y las distribuciones de los histogramas, pero ahora con una capa adicional de detalle por continente.

### Violinplot

Combina un boxplot con un gráfico de densidad. Es útil para ver no solo el resumen estadístico sino también la forma completa de la distribución.

In [ ]:
plt.figure(figsize=(12, 7))
sns.violinplot(x='continente', y='esperanza_vida', data=df)
plt.title('Distribución de Esperanza de Vida por Continente (Violin Plot)')
plt.show()

### Joint Plot

Muestra un scatter plot de dos variables y añade los histogramas o gráficos de densidad de cada variable en los márgenes.

In [ ]:
sns.jointplot(x='pib_per_capita', y='esperanza_vida', data=df_2007, alpha=0.3,marker='o')
plt.suptitle('Joint Plot de PIB per Cápita y Esperanza de Vida (2007)', y=1.02)
plt.show()

**Conclusión de este Análisis Exploratorio**

A través de este análisis estadístico, hemos extraído valiosos insights económicos sin correr un solo modelo formal:

- **Desigualdad Global:** Existe una enorme disparidad en el PIB per cápita y la esperanza de vida entre países y continentes.

- **Rendimientos Decrecientes de la Riqueza en la Salud:** La relación entre ingreso y esperanza de vida es positiva pero no lineal, sugiriendo que las políticas enfocadas en sacar a los países de la pobreza extrema tienen un impacto desproporcionadamente alto en la salud.

- **Dinámicas Continentales:** Los continentes muestran trayectorias de desarrollo muy diferentes, con una notable divergencia después de 1950 y un reciente y rápido crecimiento en Asia.

- **Calidad de los Datos:** Nuestro dataset está completo, pero en un caso real, esta fase habría sido crucial para detectar y planificar cómo manejar valores nulos o outliers problemáticos.

Ahora, con una comprensión intuitiva y profunda de la estructura y las relaciones en nuestros datos, estamos mucho mejor preparados para la siguiente fase: el machine learning.